Risk analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import folium

In [ ]:
df = pd.read_csv("/earthquake_1995-2023.csv")

print(df.head())


In [ ]:
print(df.columns)

In [ ]:
print(df.shape)

In [ ]:
#1.Analysis of Earthquake's Magnitude Distribution

plt.figure(figsize=(8,5))
plt.hist(df["magnitude"], bins=20)

plt.title("Distribution of Earthquake Magnitudes")
plt.xlabel("Magnitude")
plt.ylabel("Number of Earthquakes")

plt.show()

In [ ]:
#2.Analysis of Magnitude vs. Depth
plt.figure(figsize=(8,5))

plt.scatter(df["depth"], df["magnitude"])

plt.xlabel("Depth")
plt.ylabel("Magnitude")
plt.title("Depth vs Magnitude")

plt.show()

In [ ]:
#3.Average Magnitude
print("Average Magnitude:", df["magnitude"].mean())
print("Maximum Magnitude:", df["magnitude"].max())
print("Minimum Magnitude:", df["magnitude"].min())

In [ ]:
#4.Tsumani counts
print(df["tsunami"].value_counts())

In [ ]:
#5.Countries with most frequent earthquakes
print(df["country"].value_counts().head(10))

In [ ]:
top_countries = df["country"].value_counts().head(10)

plt.figure(figsize=(10,5))
top_countries.plot(kind="bar")

plt.title("Top 10 Countries by Earthquake Count")
plt.xlabel("Country")
plt.ylabel("Number of Earthquakes")

plt.show()

In [ ]:
#Creating a new column
df["high_risk"] = (df["magnitude"] >= 7).astype(int)     #Magnitude higher that 7 is risky




In [ ]:
#Checking for all 1000 recorded earthquakes
print(df["high_risk"].value_counts())

In [ ]:
#Giving model inputs and making a connection with high risk
X = df[["depth", "cdi", "mmi", "sig"]]

y = df["high_risk"]

In [ ]:
#Giving input of 800 data from 1000
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
#Model Training
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

model.fit(X_train, y_train)

In [ ]:
predictions = model.predict(X_test)

print(predictions[:10])

In [ ]:
accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, predictions)

print(cm)

In [ ]:
m = folium.Map(location=[0, 0], zoom_start=2)

m

In [ ]:
m = folium.Map(location=[0, 0], zoom_start=2)

for _, row in df.head(200).iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=3,
        popup=f"Magnitude: {row['magnitude']}"
    ).add_to(m)

m

In [ ]:
m = folium.Map(location=[0, 0], zoom_start=2)

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=row["magnitude"] / 2,
        popup=f"""
        Magnitude: {row['magnitude']}<br>
        Depth: {row['depth']} km<br>
        Country: {row['country']}
        """
    ).add_to(m)

m

In [ ]:
m = folium.Map(location=[0, 0], zoom_start=2)

for _, row in df.iterrows():

    color = "red" if row["high_risk"] == 1 else "blue"

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=row["magnitude"] / 2,
        color=color,
        fill=True,
        popup=f"""
        Magnitude: {row['magnitude']}<br>
        Depth: {row['depth']} km<br>
        Country: {row['country']}
        """
    ).add_to(m)

m

In [ ]:
m.save("earthquake_map.html")

Predicting earthquake

In [ ]:
df = pd.read_csv("/content/query (4).csv")

print(df.columns)
print(df.head())

In [ ]:
df["time"] = pd.to_datetime(df["time"])

print(df["time"].min())
print(df["time"].max())

In [ ]:
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month

df[["time","year","month"]].head()

In [ ]:
df["lat_grid"] = (df["latitude"] // 5) * 5
df["lon_grid"] = (df["longitude"] // 5) * 5

In [ ]:
monthly = df.groupby(
    ["lat_grid","lon_grid","year","month"]
).agg(
    eq_count=("mag","count"),
    avg_mag=("mag","mean"),
    max_mag=("mag","max"),
    avg_depth=("depth","mean")
).reset_index()

In [ ]:
monthly["major_eq"] = (monthly["max_mag"] >= 6.5).astype(int)

monthly = monthly.sort_values(
    ["lat_grid", "lon_grid", "year", "month"]
)

monthly["target"] = (
    monthly.groupby(["lat_grid", "lon_grid"])["major_eq"]
    .shift(-1)
)

In [ ]:
monthly = monthly.dropna(subset=["target"])

print(monthly.shape)

In [ ]:
print(monthly["target"].value_counts())

In [ ]:
features = [
    "eq_count",
    "avg_mag",
    "max_mag",
    "avg_depth"
]


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

print(
    importance.sort_values(
        by="Importance",
        ascending=False
    )
)

In [ ]:
monthly["prev_eq_count"] = (
    monthly.groupby(["lat_grid","lon_grid"])["eq_count"]
    .shift(1)
)

monthly["prev_max_mag"] = (
    monthly.groupby(["lat_grid","lon_grid"])["max_mag"]
    .shift(1)
)

In [ ]:
monthly["last3m_eq_count"] = (
    monthly.groupby(["lat_grid","lon_grid"])["eq_count"]
    .rolling(3)
    .sum()
    .reset_index(level=[0,1], drop=True)
)

In [ ]:
monthly = monthly.dropna()
print(monthly.shape)

In [ ]:
features = [
    "eq_count",
    "avg_mag",
    "max_mag",
    "avg_depth",
    "prev_eq_count",
    "prev_max_mag",
    "last3m_eq_count"
]

In [ ]:
X = monthly[features]
y = monthly["target"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
faults = gpd.read_file("/content/gem_active_faults.geojson")

In [ ]:
from shapely.geometry import Point

grid_points = gpd.GeoDataFrame(
    monthly,
    geometry=gpd.points_from_xy(
        monthly["lon_grid"],
        monthly["lat_grid"]
    ),
    crs="EPSG:4326"
)



In [ ]:
print(faults.geometry.geom_type.value_counts())


In [ ]:
faults_proj = faults.to_crs("EPSG:3857")
grid_proj = grid_points.to_crs("EPSG:3857")

In [ ]:
from shapely.ops import nearest_points

def nearest_fault_distance(point, faults_gdf):
    return faults_gdf.distance(point).min()

grid_proj["distance_to_fault_m"] = grid_proj.geometry.apply(
    lambda x: nearest_fault_distance(x, faults_proj)
)

In [ ]:
grid_proj["distance_to_fault_km"] = (
    grid_proj["distance_to_fault_m"] / 1000
)

print(
    grid_proj["distance_to_fault_km"].describe()
)

In [ ]:
monthly["distance_to_fault_km"] = grid_proj["distance_to_fault_km"].values

In [ ]:
features = [
    "eq_count",
    "avg_mag",
    "max_mag",
    "avg_depth",
    "prev_eq_count",
    "prev_max_mag",
    "last3m_eq_count",
    "distance_to_fault_km"
]

In [ ]:
X = monthly[features]
y = monthly["target"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))